# Train, evaluate and export a run

Run on a GPU runtime. Set the parameters in the next cell, then run the cells in order: setup, stage the shards and eval plots, throughput check, release baseline (once), train, evaluate, export, run comparison, close.

Re-running the train cell after a disconnect resumes from the last checkpoint on Drive.

In [ ]:
RUN_ID = "neon-platform"
CONFIG = "experiments/configs/neon_platform.yaml"
REPO_SHA = ""
SOURCES = ["neon"]

In [ ]:
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fair_trees")

REPO_URL = "https://github.com/OmdenaAI/HOTOSM-fAIr-Trees.git"
!git clone $REPO_URL /content/fair_trees && git -C /content/fair_trees checkout $REPO_SHA

%pip install "/content/fair_trees[train,test]" -r /content/fair_trees/experiments/requirements.txt
%cd /content/fair_trees
sys.path.insert(0, "/content/fair_trees")

In [ ]:
import torch
import torchvision

assert torch.__version__.split("+")[0] == "2.11.0" and torchvision.__version__.split("+")[0] == "0.26.0"
print(torch.cuda.get_device_name())

In [ ]:
import json
import shutil
import zipfile

from experiments.shards import stage_shards

stage_shards(DRIVE, Path("/content/data"), SOURCES)

# the eval plots come from the raw archive kept on Drive
raw_manifest = json.loads((DRIVE / "raw" / "neon" / "manifest.json").read_text())
raw_local = Path("/content/raw/neon")
with zipfile.ZipFile(DRIVE / "raw" / "neon" / raw_manifest["file"]) as z:
    z.extractall(raw_local)
for part in ("evaluation", "annotations"):
    shutil.copytree(
        raw_local / "NeonTreeEvaluation" / part,
        Path("/content/data/NeonTreeEvaluation") / part,
        dirs_exist_ok=True,
    )

In [ ]:
!python -m experiments.throughput $CONFIG $RUN_ID
throughput = json.loads((DRIVE / "runs" / RUN_ID / "throughput.json").read_text())
print(json.dumps(throughput, indent=2))
assert throughput["pass"], "loader is slower than the GPU step: stop here and report throughput.json"

Release baseline: run once. It scores the released weights on the same benchmark.

In [ ]:
from experiments import run

run.evaluate(
    "release-baseline",
    Path("experiments/benchmark.json"),
    config=Path(CONFIG),
    weights=DRIVE / "weights" / "NEON.pt",
)

In [ ]:
result = run.main(Path(CONFIG), RUN_ID)  # re-running this cell resumes
print(result.best_metrics)

In [ ]:
report = run.evaluate(RUN_ID, Path("experiments/benchmark.json"))
neon = report["parts"]["neon"]
print(neon["f1"], neon["f1_ci95"], neon["forgetting_pass"])

In [ ]:
exported = run.export(RUN_ID, Path("experiments/benchmark.json"))
print(exported)
assert exported["pass"]

The resume check compares two runs of `experiments/configs/neon_resume_check.yaml`. To produce them:

1. Set `CONFIG` to that file, run the train cell uninterrupted as `neon-resume-a`, and evaluate it.
2. Run the train cell as `neon-resume-b`. During epoch 1, after at least one 10-minute checkpoint has been written, disconnect and delete the runtime.
3. Reconnect, rerun the setup and stage cells, then rerun the same train cell with the same `RUN_ID`. Evaluate `neon-resume-b`.
4. Compare the two runs below.

In [ ]:
print(run.compare_runs("neon-resume-a", "neon-resume-b"))
print(run.compare_runs("release-baseline", RUN_ID))

In [ ]:
drive.flush_and_unmount()